# Optuna hyperparameter search (PPO + MlpPolicy, Lunar Lander)

All logic runs in this notebook. Results are written to `best_hyperparams.json` (or whatever path you set in the settings cell).

**Requirement:** run with the working directory set to the repo root `RL-LunarLander`, or set `OUTPUT` to an absolute path.


In [ ]:
from __future__ import annotations

import json
import os
import tempfile

import optuna
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from lunar_rl_common import (
    EntropyCoefScheduleCallback,
    make_ent_coef_schedule_late_linear,
    make_eval_vec_env_with_stats,
    make_ppo_lr_schedule_late_linear,
    make_train_vec_env,
    policy_kwargs,
    resolve_train_device,
    suggested_parallel_envs,
)

# --- Run settings (replaces CLI argparse) ---
N_TRIALS = 30
TIMESTEPS_PER_TRIAL = 200_000
N_EVAL_EPISODES = 10
N_ENVS = None  # None → suggested_parallel_envs()
SEED = 42
OUTPUT = "best_hyperparams.json"
ENV_ID = "LunarLander-v3"
STUDY_NAME = "ppo-lunarlander"
DEVICE_PREF = "cpu"  # "auto" | "cpu" | "cuda"

if N_ENVS is None:
    N_ENVS = suggested_parallel_envs()

device = resolve_train_device(DEVICE_PREF)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"device={device}, n_envs={N_ENVS}, output={OUTPUT!r}")


In [ ]:
def objective(trial: optuna.Trial) -> float:
    lr_start = trial.suggest_float("learning_rate", 3e-5, 3e-4, log=True)
    n_steps = trial.suggest_categorical("n_steps", [512, 1024])
    batch_size = trial.suggest_categorical("batch_size", [128, 256])
    n_epochs = trial.suggest_int("n_epochs", 3, 6)
    trial_gamma = trial.suggest_float("gamma", 0.98, 0.999)
    gae_lambda = trial.suggest_float("gae_lambda", 0.9, 0.99)
    ent_coef = trial.suggest_float("ent_coef", 1e-3, 0.03, log=True)
    target_kl = trial.suggest_float("target_kl", 0.005, 0.03, log=True)

    if batch_size > n_steps * N_ENVS:
        raise optuna.TrialPruned()

    trial_env = make_train_vec_env(
        N_ENVS, SEED, trial_gamma, env_id=ENV_ID
    )

    lr_sched = make_ppo_lr_schedule_late_linear(
        lr_start=lr_start, lr_end=5e-5, flat_until_progress=0.25
    )
    ent_sched = make_ent_coef_schedule_late_linear(
        ent_start=ent_coef, ent_end=0.002, flat_until_progress=0.25
    )
    ent_cb = EntropyCoefScheduleCallback(ent_sched)
    model = PPO(
        "MlpPolicy",
        trial_env,
        seed=SEED,
        device=device,
        verbose=0,
        policy_kwargs=policy_kwargs,
        learning_rate=lr_sched,
        clip_range=0.2,
        clip_range_vf=None,
        normalize_advantage=True,
        vf_coef=0.5,
        max_grad_norm=0.5,
        use_sde=False,
        target_kl=target_kl,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=trial_gamma,
        gae_lambda=gae_lambda,
        ent_coef=ent_coef,
    )

    model.learn(
        total_timesteps=TIMESTEPS_PER_TRIAL,
        callback=[ent_cb],
    )

    trial_vec_path = os.path.join(
        tempfile.gettempdir(), f"optuna_vecnormalize_trial_{trial.number}.pkl"
    )
    trial_env.save(trial_vec_path)
    trial_env.close()

    eval_venv = make_eval_vec_env_with_stats(trial_vec_path, SEED, env_id=ENV_ID)
    mean_reward, std_reward = evaluate_policy(
        model,
        eval_venv,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )
    eval_venv.close()
    try:
        os.remove(trial_vec_path)
    except OSError:
        pass
    del model

    score = mean_reward - std_reward
    trial.set_user_attr("mean_reward", mean_reward)
    trial.set_user_attr("std_reward", std_reward)
    return float(score)


print(
    f"Starting Optuna study: {N_TRIALS} trials, "
    f"{TIMESTEPS_PER_TRIAL:,} timesteps each, {N_ENVS} envs"
)
study = optuna.create_study(direction="maximize", study_name=STUDY_NAME)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_data = {
    "params": study.best_trial.params,
    "score": study.best_trial.value,
    "mean_reward": study.best_trial.user_attrs["mean_reward"],
    "std_reward": study.best_trial.user_attrs["std_reward"],
    "trial_number": study.best_trial.number,
    "run": {
        "seed": SEED,
        "n_envs": N_ENVS,
        "timesteps_per_trial": TIMESTEPS_PER_TRIAL,
        "n_eval_episodes": N_EVAL_EPISODES,
        "env_id": ENV_ID,
    },
}
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(best_data, f, indent=2)

print(f"\nBest trial #{study.best_trial.number}:")
print(f"  Score (mean - std): {study.best_trial.value:.2f}")
print(f"  Mean reward:        {study.best_trial.user_attrs['mean_reward']:.2f}")
print(f"  Std reward:         {study.best_trial.user_attrs['std_reward']:.2f}")
print("  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")
print(f"\nSaved to {OUTPUT!r}")
